# ⚡ Módulo 14 - Notebook 04: ETL Pipeline Producción Parquet

## 🏗️ Pipelines ETL de Clase Mundial

**Libro:** Saliendo de lo Pandito  
**Módulo:** 14 - PySpark Optimización ETL Pipelines  
**Duración estimada:** 80 minutos  
**Dificultad:** 🔴 Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos de aprendizaje

Al finalizar este notebook serás capaz de:

✅ **Diseñar** pipelines ETL de producción  
✅ **Optimizar** escritura en Parquet  
✅ **Aplicar** particionamiento inteligente  
✅ **Implementar** manejo de errores  
✅ **Construir** ETL end-to-end completo

---

## 📋 Pre-requisitos

* ✅ Módulo 14 (notebooks 01-03) completado
* ✅ Conocimiento de DataFrames y SQL
* ✅ Familiaridad con conceptos ETL

---

## 📚 Contenido

1. Arquitectura ETL en Spark
2. Formato Parquet y Optimización
3. Particionamiento de Datos
4. Manejo de Errores y Logging
5. Monitoreo y Métricas
6. Caso Integrador: ETL Completo de Ventas

---

## 💡 Por qué importa

**ETL de producción = valor empresarial:**

* 💰 **ROI:** Datos listos para análisis
* 🔒 **Confiable:** Manejo de errores robusto
* ⚡ **Rápido:** Optimizaciones aplicadas
* 📊 **Escalable:** TB/PB sin problemas

**De notebook exploratorio a sistema de producción**

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import functions as F
from datetime import datetime

print("💾 CARGANDO DATOS REALES DESDE UNITY CATALOG")
print("="*70)

CATALOG = "pandito_ds"
SCHEMA = "default"

try:
    # Cargar tabla de ventas de Los Andes Market (fuente RAW)
    df_raw = spark.table(f"{CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    
    print(f"\n✅ Datos reales cargados exitosamente")
    print(f"   📊 Registros RAW: {df_raw.count():,}")
    print(f"   📅 Período: {df_raw.select(F.min('fecha'), F.max('fecha')).collect()[0]}")
    print(f"   📍 Ubicación: Mendoza, Argentina (Los Andes Market)")
    
    # Configurar rutas de salida para ETL
    BASE_PATH = "/tmp/etl_pipeline_produccion"
    RAW_PATH = f"{BASE_PATH}/raw"
    PROCESSED_PATH = f"{BASE_PATH}/processed"
    CURATED_PATH = f"{BASE_PATH}/curated"
    
    print(f"\n🏗️ Arquitectura ETL (Capas):")
    print(f"   1. RAW:       {RAW_PATH}")
    print(f"   2. PROCESSED: {PROCESSED_PATH}")
    print(f"   3. CURATED:   {CURATED_PATH}")
    
    print(f"\n📋 Esquema de datos RAW:")
    df_raw.printSchema()
    
    print(f"\n📊 Calidad de datos (validación):")
    total = df_raw.count()
    nulls_ventas = df_raw.filter(F.col("ventas").isNull()).count()
    nulls_fecha = df_raw.filter(F.col("fecha").isNull()).count()
    
    print(f"   • Registros totales: {total:,}")
    print(f"   • Nulos en 'ventas': {nulls_ventas:,} ({100*nulls_ventas/total:.2f}%)")
    print(f"   • Nulos en 'fecha': {nulls_fecha:,} ({100*nulls_fecha/total:.2f}%)")
    
    if nulls_ventas == 0 and nulls_fecha == 0:
        print(f"   ✅ Datos limpios (sin nulos en columnas críticas)")
    else:
        print(f"   ⚠️ Se requiere limpieza de datos")
    
    print(f"\n🎯 Este notebook construirá un ETL completo:")
    print(f"   1. Extract: Leer de Unity Catalog")
    print(f"   2. Transform: Limpiar, enriquecer, agregar")
    print(f"   3. Load: Escribir en Parquet particionado")
    print(f"   4. Validar: Chequear calidad de salida")
    
    USAR_DATOS_REALES = True
    
except Exception as e:
    print(f"\n⚠️  No se pudo cargar la tabla de Unity Catalog")
    print(f"   Error: {e}")
    print(f"\n📝 Solución:")
    print(f"   1. Ejecuta primero: 00_05_Preparacion_Datos_Empresariales.ipynb")
    print(f"   2. Verifica que la tabla exista: {CATALOG}.{SCHEMA}.ventas_mensuales_mendoza_h3")
    print(f"\n   Continuando con datos sintéticos...")
    
    df_raw = None
    USAR_DATOS_REALES = False

print("\n" + "="*70)

## 📚 ETL Pipelines: De PoC a Producción

### 🏗️ Arquitectura ETL en Capas

**Patrón Medallion (Bronze/Silver/Gold):**

```
🟤 BRONZE (RAW)
   ↓ Datos crudos sin procesar
   ↓ Formato original (CSV, JSON, etc.)
   ↓
🖘️ SILVER (PROCESSED)
   ↓ Limpieza y validación
   ↓ Tipos correctos, sin duplicados
   ↓
🟡 GOLD (CURATED)
   ↓ Agregaciones y métricas
   ↓ Listo para análisis/BI
```

**Ventajas:**
* 🔄 Reproducible (cada capa puede regenerarse)
* 🔍 Debuggeable (aislar problemas por capa)
* 📊 Escalable (procesar solo lo necesario)

---

### 💾 Formato Parquet

**¿Por qué Parquet?**

| Formato | Lectura | Escritura | Tamaño | Compresión |
|---------|---------|-----------|---------|-------------|
| CSV | Lento | Rápido | Grande | No |
| JSON | Lento | Rápido | Grande | No |
| **Parquet** | **Rápido** | Medio | **Pequeño** | **Sí** |

**Ventajas de Parquet:**
* 📊 **Columnar:** Lee solo columnas necesarias
* 💾 **Compresión:** 5-10x más pequeño que CSV
* 📝 **Esquema:** Tipos embebidos
* ⚡ **Predicate Pushdown:** Filtra al leer

**Escribir Parquet:**
```python
df.write \
    .mode("overwrite") \
    .format("parquet") \
    .option("compression", "snappy") \
    .save("/path/to/output")
```

**Codecs de compresión:**
* **snappy:** Rápido (default)
* **gzip:** Mejor compresión, más lento
* **lz4:** Balance

---

### 🗂️ Particionamiento de Datos

**Particionar:** Dividir tabla en subcarpetas por columna.

**Ejemplo:**
```python
df.write \
    .mode("overwrite") \
    .partitionBy("año", "mes") \
    .parquet("/ventas")
```

**Estructura resultante:**
```
/ventas/
  año=2023/
    mes=01/
      part-00000.parquet
      part-00001.parquet
    mes=02/
      part-00000.parquet
  año=2024/
    mes=01/
      part-00000.parquet
```

**Ventaja:**
```python
# Solo lee año=2024 (ignora 2023)
df = spark.read.parquet("/ventas").filter("año = 2024")
```

**🎯 Reglas para particionar:**
* ✅ Columnas con cardinalidad baja (año, mes, zona)
* ❌ NO columnas con cardinalidad alta (id, timestamp)
* ✅ Columnas usadas frecuentemente en filtros
* ❌ NO más de 2-3 niveles (explosión de carpetas)

---

### ⚠️ Manejo de Errores

**Pipeline robusto:**

```python
try:
    # Extract
    df_raw = spark.read.csv("/input")
    
    # Validar esquema
    expected_cols = ["id", "fecha", "ventas"]
    assert all(col in df_raw.columns for col in expected_cols), "Esquema inválido"
    
    # Transform
    df_clean = df_raw \
        .filter(F.col("ventas") > 0) \
        .dropDuplicates(["id"])
    
    # Validar salida
    count = df_clean.count()
    assert count > 0, "DataFrame vacío después de limpieza"
    
    # Load
    df_clean.write.mode("overwrite").parquet("/output")
    
    print(f"✅ ETL exitoso: {count} registros procesados")
    
except Exception as e:
    print(f"❌ ETL fallido: {e}")
    # Notificar, rollback, etc.
    raise
```

---

### 📊 Monitoreo y Métricas

**Métricas clave:**

```python
import time

start_time = time.time()

# ETL aquí...

end_time = time.time()

metrics = {
    "duracion_segundos": end_time - start_time,
    "registros_input": df_raw.count(),
    "registros_output": df_clean.count(),
    "registros_descartados": df_raw.count() - df_clean.count(),
    "tamaño_output_mb": ...,  # Calcular tamaño de archivos
    "particiones": df_clean.rdd.getNumPartitions()
}

print(f"Métricas ETL: {metrics}")
```

---

### 🏗️ Pipeline ETL Completo

**Estructura:**

```python
def etl_ventas(fecha_proceso):
    """
    Pipeline ETL para procesar ventas diarias
    """
    # 1. EXTRACT
    df_raw = spark.read \
        .format("csv") \
        .option("header", "true") \
        .load(f"/raw/ventas/{fecha_proceso}/*.csv")
    
    # 2. TRANSFORM
    df_clean = df_raw \
        .filter(F.col("ventas") > 0) \
        .withColumn("año", F.year("fecha")) \
        .withColumn("mes", F.month("fecha")) \
        .dropDuplicates(["transaccion_id"])
    
    # 3. AGGREGATE (Gold)
    df_agg = df_clean \
        .groupBy("año", "mes", "zona") \
        .agg(
            F.sum("ventas").alias("ventas_totales"),
            F.count("*").alias("transacciones"),
            F.avg("ventas").alias("ticket_promedio")
        )
    
    # 4. LOAD (Silver)
    df_clean.write \
        .mode("append") \
        .partitionBy("año", "mes") \
        .parquet("/silver/ventas")
    
    # 5. LOAD (Gold)
    df_agg.write \
        .mode("overwrite") \
        .partitionBy("año", "mes") \
        .parquet("/gold/ventas_agregadas")
    
    return {
        "status": "success",
        "registros_procesados": df_clean.count()
    }

# Ejecutar
result = etl_ventas("2024-01-15")
print(result)
```

---

### 🎯 Best Practices

**1️⃣ Idempotencia**
```python
# Mismo input → mismo output (sin efectos secundarios)
df.write.mode("overwrite").parquet("/output")  # Sobrescribe, no acumula
```

**2️⃣ Incremental vs Full Load**
```python
# Incremental (solo nuevos datos)
df_new = df_raw.filter(f"fecha = '{fecha_proceso}'")

# Full Load (recalcula todo)
df_all = df_raw  # Re-procesa histórico completo
```

**3️⃣ Validación de calidad**
```python
# Post-load validation
df_output = spark.read.parquet("/output")
assert df_output.count() == expected_count
assert df_output.filter("ventas < 0").count() == 0
```

**4️⃣ Logging detallado**
```python
import logging

logger = logging.getLogger(__name__)
logger.info(f"Iniciando ETL para {fecha_proceso}")
logger.info(f"Registros leídos: {df_raw.count()}")
```

**5️⃣ Optimización final**
```python
# Consolidar archivos pequeños antes de escribir
df.coalesce(10).write.parquet("/output")  # 10 archivos en lugar de 200
```

In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
import time
import warnings
warnings.filterwarnings('ignore')

print("🏗️ ETL PIPELINE PRODUCCIÓN PARQUET")
print("="*70)

print(f"\nVersión de Pandas: {pd.__version__}")
print(f"Versión de NumPy: {np.__version__}")

try:
    print(f"Versión de Spark: {spark.version}")
except:
    print("⚠️  SparkSession no disponible")

print("\n🎯 En este notebook aprenderás:")
print("  • Arquitectura Medallion (Bronze/Silver/Gold)")
print("  • Escribir Parquet con compresión")
print("  • Particionar datos (partitionBy)")
print("  • Manejo de errores robusto")
print("  • Monitoreo y métricas de ETL")

print("\n📖 Métodos clave:")
print("  - df.write.format('parquet').save(path)")
print("  - df.write.partitionBy('año', 'mes')")
print("  - df.write.mode('overwrite' | 'append')")
print("  - df.write.option('compression', 'snappy')")
print("  - spark.read.parquet(path)")

print("\n" + "="*70)
print("✅ Librerías cargadas correctamente")

In [0]:
# Código de inicialización de notebook reindexado
import pandas as pd
import numpy as np
print('Notebook reindexado listo para práctica en Databricks')